# 15 -- Leaderboard agent: closing the gap to top replays

Triggered by a real leaderboard replay, `replays/103962387.json`
(`ringbearer` 105735 vs `islet` 135063, both far above our current best
`melon_then_wheat_goose_cow_agent` ~31-32k). Full action-count analysis in
chat and `PROGRESS.md`. Both players use an almost identical strategy --
a strong signal this is close to "the meta", not an outlier.

**Explicit scope decision from chat**: NOT cloning the opponent's replay
trace (see the `kaggriculture-hamburger.ipynb` discussion in
`RESOURCES.md` -- that notebook literally replays another named
competitor's fixed 720-action sequence from one past episode, doesn't
read `obs` at all, and isn't something this project uses). This notebook
infers *generalizable rules* from the replay's action patterns (same
methodology as every other opponent-replay entry in `PROGRESS.md`), then
builds our own version of that logic from the engine, same as always.

**Build order, from the replay's own day-by-day timeline** (extracted in
chat -- first occurrence of each action, both seats, nearly identical
days for both -- see the table logged in `PROGRESS.md`), not a
from-scratch priority guess. One at a time, each validated against the
previous best via `evaluate.py` before starting the next -- mixing
changes makes it impossible to attribute a regression to the right one
(burned us once already, `08_melon_hand.ipynb`):

1. **`STRAWBERRY`** (day 5 in the replay) -- first new thing after the
   animals we already have. `ongoing: True`, genuinely new harvest
   mechanic, not a copy of `WHEAT`/`MELON`'s.
2. **Land + more hands, fixed day schedule** (day 6 in the replay --
   both seats buy `NE` on the *same* step regardless of money, not
   gated on affordability like `melon_then_wheat_extra_hands_agent`).
   Moved up in our order (was last) because it opens NW-adjacent room
   that step 3 needs.
3. **`SHEEP`** (replay: day 0, but we're keeping our day-0 `GOOSE` as-is
   and adding this after land, since NE tiles solve the "no free NW
   zone left" problem from `notebooks/14_cow.ipynb` for free).
4. **`FERTILIZE`, applied not just sold** (day 14 in the replay).
5. **`DIG`** (first used day 7-12 in the replay, opportunistic, low
   risk/isolated -- reclaim `WEED` tiles instead of leaving them dead).
6. **`CARROT`** (replay: day 21-25, almost the last thing they do, ~1-4
   plantings total per hand across the whole game -- genuinely marginal
   for these top players, so it's marginal for us too).

Note: the replay builds `GOOSE`/`COOP` on day 10, late -- we already have
it validated from day 0 (`melon_then_wheat_goose_cow_agent`). Not
reworking a working, validated win just for order fidelity.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import (
    DIRECTIONS,
    GOOSE_FEEDER,
    GOOSE_TILE,
    HAND_ZONES,
    LOOP_ORDER_12,
    LOOP_ORDER_HANDS,
    PASTURE_FEEDER,
    PASTURE_TILE,
    SHED_ADJACENT,
    SPAWN_FEEDER,
    melon_then_wheat_goose_cow_agent,
)

## Reference: the analyzed replay

Quick re-load, in case any number above needs double-checking against
the real data instead of the chat summary.

In [2]:
import json

with open(Path.cwd().parent / "replays" / "103962387.json", encoding="utf-8") as f:
    top_replay = json.load(f)

print(top_replay["info"]["Agents"])
print("final rewards:", top_replay["rewards"])

[{'Name': 'ringbearer', 'ThumbnailUrl': None}, {'Name': 'islet', 'ThumbnailUrl': None}]
final rewards: [105735.0, 135063.0]


## 1. `STRAWBERRY` -- `ongoing` crop, genuinely new harvest mechanic

Confirmed via engine source (`_daily_refresh_plants`, session before this
one): production accumulates daily via `interval` (like an animal), the
tile never clears after `HARVEST`, and critically -- **any** planted tile
(ongoing or not) turns to `WEED` after 2 consecutive unwatered days. Our
current `if crop_age >= threshold: HARVEST else: WATER` pattern would
stop watering forever once harvest starts (never revisits the `WATER`
branch for a tile that's already "ready") -- would kill the plant. Needs
water-and-harvest-both logic, not an `elif`.

Open question to resolve before writing code: which hand(s) grow it?
Replay data (chat): most hands plant `STRAWBERRY` a handful of times each,
mixed in with `WHEAT` on the same zone -- not one dedicated hand.

In [21]:
CROP_INFO = {
    "WHEAT": {"ongoing": False, "max_yield_day": 4, "max_yield": 6},
    "MELON": {"ongoing": False, "max_yield_day": 12, "max_yield": 6},
    "STRAWBERRY": {"ongoing": True},
}
STRAWBERRY_TILE = (0,1)

def _tile_action(tile, fallback, crop_seeds, crop_to_plant, obs):

    if tile is None and crop_seeds > 0:
        action = ["PLANT", crop_to_plant]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        ongoing_crop = CROP_INFO[tile["crop"]]["ongoing"]
        if ongoing_crop:
            if not tile.get("watered_today"):
                action = ["WATER"]
            elif tile.get("yield_units", 0) > 0:
                action = ["HARVEST"]
            else: 
                action = fallback
        else:
            max_yield_day = CROP_INFO[tile["crop"]]["max_yield_day"]
            max_yield = CROP_INFO[tile["crop"]]["max_yield"]
            if crop_age >= max_yield_day or tile.get("yield_units",0) >= max_yield:
                action = ["HARVEST"]
            elif not tile.get("watered_today"):
                action = ["WATER"]
            else: 
                action = fallback
    else:
        action = fallback
    return action

In [20]:
def strawberry_agent(obs):
    """Skeleton, copy of melon_then_wheat_goose_cow_agent -- fill in the
    STRAWBERRY logic at the TODO markers below (notebooks/15_leaderboard_agent.ipynb, step 1).
    As-is (no TODOs filled in yet) this should behave identically to
    melon_then_wheat_goose_cow_agent -- a good first check before changing
    anything.
    """
    crop_to_plant = "MELON" if obs["day"] < 10 else "WHEAT"
    crop_seeds = obs["private"]["seeds"].get(crop_to_plant, 0)
    strawberry_seeds = obs["private"]["seeds"].get("STRAWBERRY", 0)

    market = []
    hands = []

    if obs["step"] == 0:
        market.append(["BUY_ANIMAL", "GOOSE", 1])
        market.append(["BUY_ANIMAL", "COW", 1])
        market.append(["BUY_SEED","STRAWBERRY", 1])

    if obs["day"] < 13 and obs["private"]["shed"].get("WHEAT", 0) == 0:
        market.append(["BUY_PRODUCT", "WHEAT", 1])

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        if i == 4:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **GOOSE_FEEDER}
        elif i == 2:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **PASTURE_FEEDER}
        else:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]

        if i == 4:
            hand_inventory = obs["private"]["inventories"][i + 1]

            if (hx, hy) == GOOSE_TILE:
                if hand_tile is None:
                    op = ["BUILD_COOP"]
                elif hand_tile.get("kind") == "COOP" and hand_tile.get("animal") != "GOOSE":
                    if hand_inventory.get("GOOSE", 0) > 0:
                        op = ["PLACE", "GOOSE", 1]
                    else:
                        op = ["PASS"]
                elif hand_inventory.get("WHEAT", 0) > 0 and not hand_tile.get("fed_today"):
                    op = ["FEED"]
                elif hand_tile.get("fertilizer_available"):
                    op = ["COLLECT_FERTILIZER"]
                elif not hand_tile.get("cared_today"):
                    op = ["CARE"]
                elif hand_tile.get("yield_units", 0) > 0:
                    op = ["HARVEST"]
                else:
                    op = ["PASS"]
                    egg_in_shed = obs["private"]["shed"].get("EGG", 0)
                    if egg_in_shed > 0:
                        market.append(["SELL", "EGG", egg_in_shed])
            else:
                if hand_tile == "LOCKED":
                    op = move
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("GOOSE", 0) == 0 and obs["private"]["shed"].get("GOOSE", 0) > 0:
                    op = ["PICKUP", "GOOSE", 1]
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("WHEAT", 0) == 0 and obs["private"]["shed"].get("WHEAT", 0) > 0:
                    op = ["PICKUP", "WHEAT", 1]
                else:
                    op = _tile_action(hand_tile, move, crop_seeds, crop_to_plant, obs)
        elif i == 2:
            hand_inventory = obs["private"]["inventories"][i + 1]

            if (hx, hy) == PASTURE_TILE:
                if hand_tile is None:
                    op = ["BUILD_PASTURE"]
                elif hand_tile.get("kind") == "PASTURE" and hand_tile.get("animal") != "COW":
                    if hand_inventory.get("COW", 0) > 0:
                        op = ["PLACE", "COW", 1]
                    else:
                        op = ["PASS"]
                elif hand_inventory.get("WHEAT", 0) > 0 and not hand_tile.get("fed_today"):
                    op = ["FEED"]
                elif hand_tile.get("fertilizer_available"):
                    op = ["COLLECT_FERTILIZER"]
                elif not hand_tile.get("cared_today"):
                    op = ["CARE"]
                elif hand_tile.get("yield_units", 0) > 0:
                    op = ["HARVEST"]
                else:
                    op = ["PASS"]
            else:
                if hand_tile == "LOCKED":
                    op = move
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("COW", 0) == 0 and obs["private"]["shed"].get("COW", 0) > 0:
                    op = ["PICKUP", "COW", 1]
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("WHEAT", 0) == 0 and obs["private"]["shed"].get("WHEAT", 0) > 0:
                    op = ["PICKUP", "WHEAT", 1]
                else:
                    op = _tile_action(hand_tile, move, crop_seeds, crop_to_plant, obs)
        elif hand_tile == "LOCKED" or (hx, hy) not in HAND_ZONES[i % len(HAND_ZONES)]:
            op = move
        else:
            if (hx, hy) == STRAWBERRY_TILE:
                op = _tile_action(hand_tile, move, strawberry_seeds,"STRAWBERRY", obs)
            else:
                op = _tile_action(hand_tile, move, crop_seeds, crop_to_plant, obs)
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or crop_seeds <= 5:
        market.append(["BUY_SEED", crop_to_plant, 6 - crop_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    fertilizer_in_shed = obs["private"]["shed"].get("FERTILIZER", 0)
    milk_in_shed = obs["private"]["shed"].get("MILK", 0)
    strawberry_in_shed = obs["private"]["shed"].get("STRAWBERRY", 0)
    if wheat_in_shed > 1:
        market.append(["SELL", "WHEAT", wheat_in_shed - 1])
    if melon_in_shed > 0:
        market.append(["SELL", "MELON", melon_in_shed])
    if fertilizer_in_shed > 0:
        market.append(["SELL", "FERTILIZER", fertilizer_in_shed])
    if milk_in_shed > 0:
        market.append(["SELL", "MILK", milk_in_shed])
    if strawberry_in_shed > 0:
            market.append(["SELL", "STRAWBERRY", strawberry_in_shed])
    farmer = _tile_action(tile, [direction], crop_seeds, crop_to_plant, obs)

    return {"farmer": farmer, "hands": hands, "market": market}

In [22]:
evaluate(melon_then_wheat_goose_cow_agent, strawberry_agent, n=10)

{'win_rate': 0.0, 'avg_money': 31778.0}

In [23]:
evaluate(melon_then_wheat_goose_cow_agent, strawberry_agent, n=10)

{'win_rate': 0.0, 'avg_money': 29762.0}

## 2. Land + more hands, fixed day schedule

Not started yet. Replay: both seats buy `NE` on the exact same step
(151, day 6) regardless of their own money at that point -- a day-based
trigger, not the affordability gate `melon_then_wheat_extra_hands_agent`
uses. Also reaches 11-12 hands total (we cap at 8).

In [4]:
# TODO: not started.

## 3. `SHEEP` -- second `PASTURE` animal

Not started yet. Confirmed identical mechanism to `COW`
(`notebooks/14_cow.ipynb`'s engine-source finding). With NE land from
step 2 above, `SHEEP` gets a fresh tile there instead of needing to steal
another NW zone from hands 0/1/3.

In [5]:
# TODO: not started.

## 4. `FERTILIZE` -- apply instead of only selling

Not started yet. Consumes 1 `FERTILIZER` from the acting unit's own
inventory, standing on a `PLANT` tile, doubles that tile's yield for the
next 3 days (`fertilized_until_day`). We currently only `SELL FERTILIZER`
(2026-08-31 session); the replay's top players mostly apply it (day 14).

In [6]:
# TODO: not started.

## 5. `DIG` -- reclaim `WEED`/empty structure tiles

Not started yet. New action, not previously known to us: clears a tile
back to `None` (plant, weed, or empty `COOP`/`PASTURE` -- NOT a placed
animal). Used opportunistically in the replay (first seen day 7-12, not
a heavy pattern) -- low risk, isolated addition.

In [7]:
# TODO: not started.

## 6. `CARROT` -- a fourth crop, last on purpose

Not started yet. Replay data (chat): genuinely marginal for these top
players -- only 1-4 plantings per hand across the *entire* game, first
one as late as day 21-25, scattered across tiles with no fixed zone.
Same `ongoing: False` mechanic already built for `WHEAT`/`MELON` (no new
harvest logic needed), just low priority -- build it last, and lightly.

In [8]:
# TODO: not started.